# Analisis de Hiperparametros NER

Este notebook analiza los resultados del grid search de NER guardados en `../artifacts/ner/hyperparam_results.json`.

El objetivo es comparar configuraciones, identificar que hiperparametros afectan mas al rendimiento y decidir que modelo conviene conservar como mejor candidato usando la metrica principal de entity F1.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

ROOT_DIR = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "analysis"
    else Path.cwd().resolve()
)
OUTPUT_DIR = ROOT_DIR / "analysis" / "artifact_ner_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga de resultados


In [ ]:
results_path = ROOT_DIR / "artifacts" / "ner" / "hyperparam_results.json"
if not results_path.exists():
    raise FileNotFoundError(f"No existe {results_path}")

results = json.loads(results_path.read_text(encoding="utf-8"))
if not isinstance(results, list) or not results:
    raise ValueError("hyperparam_results.json esta vacio o no contiene una lista")

print(f"Total de configuraciones registradas: {len(results)}")
print("Claves del primer resultado:", sorted(results[0].keys()))
print("Claves de config:", sorted(results[0]["config"].keys()))

## 2. Preparar DataFrame


In [ ]:
rows = []
for item in results:
    row = dict(item["config"])
    row["run_dir"] = item.get("run_dir")
    row["best_val_loss"] = item.get("best_val_loss")
    row["best_val_token_accuracy"] = item.get("best_val_token_accuracy")
    row["best_val_entity_token_f1_micro"] = item.get("best_val_entity_token_f1_micro")
    row["best_epoch"] = item.get("best_epoch")
    row["status"] = item.get("status")
    row["error"] = item.get("error")
    rows.append(row)

df = pd.DataFrame(rows)
df["freeze_backbone"] = df["freeze_backbone"].astype(str)
df.head()

In [ ]:
completed = df[df["status"] == "completed"].copy()
failed = df[df["status"] != "completed"].copy()

print(f"Configuraciones completadas: {len(completed)}")
print(f"Configuraciones fallidas: {len(failed)}")

if not failed.empty:
    display(
        failed[["batch_size", "epochs", "learning_rate", "freeze_backbone", "error"]]
    )

## 3. Ranking de mejores configuraciones


In [ ]:
ranking = completed.sort_values(
    by=["best_val_entity_token_f1_micro", "best_val_loss"],
    ascending=[False, True],
).reset_index(drop=True)

ranking[
    [
        "batch_size",
        "epochs",
        "learning_rate",
        "freeze_backbone",
        "best_epoch",
        "best_val_entity_token_f1_micro",
        "best_val_loss",
        "run_dir",
    ]
].head(10)

In [ ]:
best = ranking.iloc[0]
print("Mejor configuracion encontrada:")
print(f"  batch_size={best['batch_size']}")
print(f"  epochs={best['epochs']}")
print(f"  learning_rate={best['learning_rate']}")
print(f"  freeze_backbone={best['freeze_backbone']}")
print(f"  best_epoch={best['best_epoch']}")
print(f"  val_entity_token_f1_micro={best['best_val_entity_token_f1_micro']:.4f}")
print(f"  val_loss={best['best_val_loss']:.4f}")
print(f"  run_dir={best['run_dir']}")

In [ ]:
top10_plot = ranking.head(10).copy()
labels = [
    f"bs={row.batch_size}, ep={row.epochs}, lr={row.learning_rate}, freeze={row.freeze_backbone}"
    for row in top10_plot.itertuples()
]
plt.figure(figsize=(10, 5))
sns.barplot(data=top10_plot, x="best_val_entity_token_f1_micro", y=labels, orient="h")
plt.title("Top 10 configuraciones NER")
plt.xlabel("best_val_entity_token_f1_micro")
plt.ylabel("configuracion")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top10_ner_configs.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Resumen por hiperparametro


In [ ]:
summary_by_freeze = (
    completed.groupby("freeze_backbone")
    .agg(
        mean_f1=("best_val_entity_token_f1_micro", "mean"),
        max_f1=("best_val_entity_token_f1_micro", "max"),
        mean_loss=("best_val_loss", "mean"),
        count=("best_val_entity_token_f1_micro", "count"),
    )
    .sort_values("max_f1", ascending=False)
)
summary_by_freeze

In [ ]:
summary_by_lr = (
    completed.groupby("learning_rate")
    .agg(
        mean_f1=("best_val_entity_token_f1_micro", "mean"),
        max_f1=("best_val_entity_token_f1_micro", "max"),
        mean_loss=("best_val_loss", "mean"),
        count=("best_val_entity_token_f1_micro", "count"),
    )
    .sort_values("max_f1", ascending=False)
)
summary_by_lr

In [ ]:
summary_by_batch = (
    completed.groupby("batch_size")
    .agg(
        mean_f1=("best_val_entity_token_f1_micro", "mean"),
        max_f1=("best_val_entity_token_f1_micro", "max"),
        mean_loss=("best_val_loss", "mean"),
        count=("best_val_entity_token_f1_micro", "count"),
    )
    .sort_values("max_f1", ascending=False)
)
summary_by_batch

In [ ]:
summary_by_epochs = (
    completed.groupby("epochs")
    .agg(
        mean_f1=("best_val_entity_token_f1_micro", "mean"),
        max_f1=("best_val_entity_token_f1_micro", "max"),
        mean_loss=("best_val_loss", "mean"),
        count=("best_val_entity_token_f1_micro", "count"),
    )
    .sort_values("max_f1", ascending=False)
)
summary_by_epochs

## 5. Graficas


In [ ]:
plt.figure()
sns.boxplot(data=completed, x="freeze_backbone", y="best_val_entity_token_f1_micro")
plt.title("F1 por freeze_backbone")
plt.xlabel("freeze_backbone")
plt.ylabel("best_val_entity_token_f1_micro")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "f1_by_freeze_backbone.png", dpi=180)
plt.show()

In [ ]:
plt.figure()
sns.boxplot(data=completed, x="batch_size", y="best_val_entity_token_f1_micro")
plt.title("F1 por batch_size")
plt.xlabel("batch_size")
plt.ylabel("best_val_entity_token_f1_micro")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "f1_by_batch_size.png", dpi=180)
plt.show()

In [ ]:
plt.figure()
sns.boxplot(data=completed, x="epochs", y="best_val_entity_token_f1_micro")
plt.title("F1 por numero de epocas")
plt.xlabel("epochs")
plt.ylabel("best_val_entity_token_f1_micro")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "f1_by_epochs.png", dpi=180)
plt.show()

In [ ]:
plt.figure()
sns.boxplot(data=completed, x="learning_rate", y="best_val_entity_token_f1_micro")
plt.title("F1 por learning_rate")
plt.xlabel("learning_rate")
plt.ylabel("best_val_entity_token_f1_micro")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "f1_by_learning_rate.png", dpi=180)
plt.show()

In [ ]:
pivot = completed.pivot_table(
    index="batch_size",
    columns="learning_rate",
    values="best_val_entity_token_f1_micro",
    aggfunc="mean",
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("Entity F1 medio: batch_size vs learning_rate")
plt.ylabel("batch_size")
plt.xlabel("learning_rate")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_batch_lr_f1.png", dpi=180)
plt.show()

## 6. Conclusiones rapidas


In [ ]:
top5 = ranking.head(5)[
    [
        "batch_size",
        "epochs",
        "learning_rate",
        "freeze_backbone",
        "best_epoch",
        "best_val_entity_token_f1_micro",
        "best_val_loss",
    ]
]
top5

In [ ]:
print("Mejor opcion por entity F1:")
print(top5.iloc[0].to_string())

summary_lines = [
    "# Analisis NER",
    f"- Configuraciones registradas: {len(results)}",
    f"- Configuraciones completadas: {len(completed)}",
    f"- Mejor batch_size: {best['batch_size']}",
    f"- Mejor epochs: {best['epochs']}",
    f"- Mejor learning_rate: {best['learning_rate']}",
    f"- Mejor freeze_backbone: {best['freeze_backbone']}",
    f"- Mejor best_epoch: {best['best_epoch']}",
    f"- Mejor val_entity_token_f1_micro: {best['best_val_entity_token_f1_micro']:.6f}",
    f"- Mejor val_loss: {best['best_val_loss']:.6f}",
]
(OUTPUT_DIR / "summary.md").write_text(
    "\n".join(summary_lines) + "\n", encoding="utf-8"
)
print(
    "\nSugerencia: elegir la configuracion con mayor val_entity_token_f1_micro y usar val_loss como desempate."
)
print(f"Resumen guardado en {OUTPUT_DIR / 'summary.md'}")